<a href="https://colab.research.google.com/github/jackeywangxinyu/learnAngular/blob/main/lp_elf_jepa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lp-ELF-JEPA Trial Ground

This notebook implements a small, stable, single-GPU JEPA-style latent world model and a visual trial ground.
Inspired by VL-JEPA, this prototype replaces pixel reconstruction with an Lp-JEPA / Rectified-LpJEPA visual representation learner, and replaces autoregressive language modeling with an ELF-style continuous language embedding flow (Mini-ELF).

It utilizes a SCOTT-lite convolutional tokenizer to improve small-data stability, and trains an action-conditioned latent dynamics model for planning.

**Research Goals Demonstrated:**
1. Visual representation space does not collapse (verified via SIGReg-lite metrics).
2. Visual latents align with continuous language semantic embeddings.
3. The model predicts next visual latents after actions.
4. A latent planner beats a random policy in a narrow-domain toy world.
5. The language side operates in a continuous flow, skipping autoregressive token generation.


In [ ]:
# Section 2: Environment setup
!pip install -q torch torchvision transformers sentencepiece einops tqdm pandas matplotlib numpy scikit-learn Pillow imageio imageio-ffmpeg


In [ ]:
# Section 3: Imports, reproducibility, and runtime checks
import os
import math
import random
import warnings
from dataclasses import dataclass, field
from typing import Optional, Tuple, List, Dict, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from transformers import T5EncoderModel, T5Tokenizer
from einops import rearrange, repeat
import matplotlib.pyplot as plt
import pandas as pd
import imageio
from tqdm.auto import tqdm
from IPython.display import display, HTML, Video

warnings.filterwarnings("ignore")

def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Runtime Information:")
!nvidia-smi
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f"CUDA Available. Device: {torch.cuda.get_device_name(0)}")
    print("TF32 enabled for Ampere+ GPUs.")
else:
    print("Running on CPU. Training will be slow.")


Runtime Information:
Thu May 14 03:51:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P0             44W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+--------------------------

In [ ]:
# Section 4: Configuration
@dataclass
class Config:
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    output_dir: str = "./outputs"

    # Env & Data
    image_size: int = 64
    grid_size: int = 8
    num_train_samples: int = 5000
    num_val_samples: int = 500
    max_text_len: int = 32
    num_actions: int = 4  # movement only: up/down/left/right
    action_names: List[str] = field(default_factory=lambda: ["up", "down", "left", "right"])
    num_workers: int = 0  # DataLoader workers; >0 parallelizes the on-the-fly env rollouts

    # Model architecture
    visual_embed_dim: int = 128
    visual_depth: int = 4
    visual_heads: int = 4
    visual_mlp_ratio: int = 4
    patch_size: int = 16
    mask_ratio: float = 0.5

    text_model_name: str = "t5-small"
    text_embed_dim: int = 512
    text_bottleneck_dim: int = 128

    mini_elf_width: int = 128
    mini_elf_depth: int = 3
    mini_elf_heads: int = 4

    dynamics_depth: int = 3

    # Training
    batch_size: int = 32
    grad_accum_steps: int = 1
    lr: float = 3e-4
    weight_decay: float = 0.05
    warmup_steps: int = 100

    stage1_steps: int = 2000
    stage2_steps: int = 500
    stage3_steps: int = 500
    stage4_steps: int = 1500

    use_bf16: bool = True if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else False
    use_tf32: bool = True
    use_compile: bool = False

    anti_collapse_mode: str = "sigreg_lite"
    lambda_pred: float = 1.0
    lambda_std: float = 0.1
    lambda_cov: float = 0.1
    lambda_sparse: float = 0.0
    lambda_align: float = 1.0
    lambda_infonce: float = 0.1
    lambda_dynamics: float = 1.0
    lambda_decode: float = 0.0

    ema_decay: float = 0.99

    # Trial Ground & Planning
    planner_horizon: int = 4
    planner_num_candidates: int = 64
    trial_max_steps: int = 20
    render_size: int = 128
    save_videos: bool = True
    config_name: str = "H100"

SMOKE_CONFIG = Config(
    config_name="SMOKE",
    num_train_samples=500, num_val_samples=50,
    stage1_steps=200, stage2_steps=100, stage3_steps=100, stage4_steps=200,
    batch_size=8, visual_embed_dim=64, visual_depth=2, visual_heads=4,
    mini_elf_width=64, mini_elf_depth=2, text_bottleneck_dim=64, dynamics_depth=2,
    num_workers=0,
)

H100_CONFIG = Config()  # baseline defaults (mid-tier)

# A100-class run — sized to actually load the GPU, not waste it.
# - image_size 128 → 64 patches/image (4x more tokens than the old 64x64)
# - bigger model (512-dim, 8-deep) so each step does real work
# - batch 384 + 4 DataLoader workers so the GPU is fed continuously
# - planner is vectorized across candidates, no Python-serial rollouts
A100_CONFIG = Config(
    config_name="A100",
    image_size=128,
    num_train_samples=50000, num_val_samples=2000,
    batch_size=384,
    num_workers=4,
    visual_embed_dim=512, visual_depth=8, visual_heads=8, visual_mlp_ratio=4,
    mini_elf_width=384, mini_elf_depth=4, mini_elf_heads=8,
    text_bottleneck_dim=256, dynamics_depth=4,
    stage1_steps=6000, stage2_steps=2000, stage3_steps=2000, stage4_steps=4000,
    planner_num_candidates=256, planner_horizon=6,
)

# Choose config — default to A100. Switch to SMOKE_CONFIG for a fast CPU sanity-check.
config = A100_CONFIG
seed_everything(config.seed)
os.makedirs(config.output_dir, exist_ok=True)
print(f"Active config: {config.config_name}  |  device: {config.device}  |  bf16: {config.use_bf16}")
print(f"image_size: {config.image_size}  batch: {config.batch_size}  workers: {config.num_workers}")
print(f"visual: {config.visual_embed_dim}d x{config.visual_depth}  |  Stage1/4 steps: {config.stage1_steps}/{config.stage4_steps}")


Active config: A100  |  device: cuda  |  bf16: True
image_size: 128  batch: 384  workers: 4
visual: 512d x8  |  Stage1/4 steps: 6000/4000


In [ ]:
# Section 5: Procedural BlockWorld dataset
class BlockWorldEnv:
    def __init__(self, size: int = 8, image_size: int = 64):
        self.size = size
        self.image_size = image_size
        # 4 movement actions: up, down, left, right
        self.action_map = {0: (0,-1), 1: (0,1), 2: (-1,0), 3: (1,0)}

        self.colors = {
            'bg': [200, 200, 200], 'agent': [0, 0, 255], 'goal': [0, 255, 0],
            'lava': [255, 0, 0], 'key': [255, 255, 0], 'door': [139, 69, 19]
        }

    def reset(self, seed: Optional[int] = None):
        if seed is not None: np.random.seed(seed)
        self.agent_pos = [np.random.randint(0, self.size), np.random.randint(0, self.size)]
        self.goal_pos = [np.random.randint(0, self.size), np.random.randint(0, self.size)]
        while self.goal_pos == self.agent_pos:
            self.goal_pos = [np.random.randint(0, self.size), np.random.randint(0, self.size)]

        self.lava_pos = [np.random.randint(0, self.size), np.random.randint(0, self.size)]
        self.has_key = False
        return self._get_obs()

    def step(self, action_id: int):
        dx, dy = self.action_map[action_id]
        new_x = np.clip(self.agent_pos[0] + dx, 0, self.size - 1)
        new_y = np.clip(self.agent_pos[1] + dy, 0, self.size - 1)

        self.agent_pos = [new_x, new_y]
        success = (self.agent_pos == self.goal_pos)

        goal_text = "reach the green goal without stepping into lava"
        state_text = f"agent is at {self.agent_pos[0]},{self.agent_pos[1]}. goal is at {self.goal_pos[0]},{self.goal_pos[1]}"
        rationale = "move closer to the goal" if not success else "goal reached"
        if self.agent_pos == self.lava_pos: rationale = "agent stepped in lava"

        return self._get_obs(), success, goal_text, state_text, rationale

    def _render(self, agent_pos):
        """Render a frame with the agent at the given position (helper)."""
        img = np.ones((self.size, self.size, 3), dtype=np.uint8) * np.array(self.colors['bg'], dtype=np.uint8)
        img[self.lava_pos[1], self.lava_pos[0]] = self.colors['lava']
        img[self.goal_pos[1], self.goal_pos[0]] = self.colors['goal']
        img[agent_pos[1], agent_pos[0]] = self.colors['agent']
        img = img.repeat(self.image_size // self.size, axis=0).repeat(self.image_size // self.size, axis=1)
        return rearrange(torch.tensor(img, dtype=torch.float32) / 255.0, 'h w c -> c h w')

    def _get_obs(self):
        return self._render(self.agent_pos)

    def get_goal_frame(self):
        """Render the success state: agent rendered at the goal cell (no separate goal cell)."""
        return self._render(self.goal_pos)

class ProceduralDataset(Dataset):
    """On-the-fly generation: each __getitem__ builds one (frame_t, action, frame_tp1, goal_frame)
    transition from a deterministic seed. This avoids holding the whole dataset in host RAM
    when image_size is large and lets num_workers parallelize the env rollouts."""
    def __init__(self, num_samples: int, config: Config, seed_offset: int = 0):
        self.num_samples = int(num_samples)
        self.config = config
        self.seed_offset = int(seed_offset)

    def __len__(self): return self.num_samples

    def __getitem__(self, idx):
        seed = idx + self.seed_offset
        env = BlockWorldEnv(size=self.config.grid_size, image_size=self.config.image_size)
        frame_t = env.reset(seed=seed)
        goal_frame = env.get_goal_frame()
        action = int(np.random.randint(0, self.config.num_actions))
        frame_tp1, success, g_text, s_text, r_text = env.step(action)
        return {
            'frame_t': frame_t, 'frame_tp1': frame_tp1, 'action_id': action,
            'goal_frame': goal_frame,
            'goal_text': g_text, 'state_text': s_text, 'rationale_text': r_text,
            'success': success,
        }

train_dataset = ProceduralDataset(config.num_train_samples, config, seed_offset=0)
val_dataset = ProceduralDataset(config.num_val_samples, config, seed_offset=10**6)

_pin = (config.device == 'cuda')
_persist = config.num_workers > 0
train_loader = DataLoader(
    train_dataset, batch_size=config.batch_size, shuffle=True, drop_last=True,
    num_workers=config.num_workers, pin_memory=_pin, persistent_workers=_persist,
)
val_loader = DataLoader(
    val_dataset, batch_size=config.batch_size, shuffle=False,
    num_workers=config.num_workers, pin_memory=_pin, persistent_workers=_persist,
)
print(f"Dataset ready: train={len(train_dataset)}, val={len(val_dataset)} (on-the-fly, {config.num_workers} workers)")


Dataset ready: train=50000, val=2000 (on-the-fly, 4 workers)


In [ ]:
# Section 6: SCOTT-lite convolutional tokenizer
class SCOTTLiteTokenizer(nn.Module):
    def __init__(self, in_channels=3, embed_dim=128, patch_size=16):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Sequential(
            nn.Conv2d(in_channels + 1, 32, kernel_size=3, stride=2, padding=1),
            nn.GroupNorm(8, 32),
            nn.SiLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.GroupNorm(16, 64),
            nn.SiLU(),
            nn.Conv2d(64, embed_dim, kernel_size=patch_size//4, stride=patch_size//4),
            nn.GroupNorm(16, embed_dim) if embed_dim >= 16 else nn.Identity()
        )
        self.out_dim = embed_dim

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None):
        B, C, H, W = x.shape
        P = self.patch_size
        N_H, N_W = H // P, W // P

        if mask is None:
            mask = torch.ones((B, N_H * N_W), device=x.device)

        spatial_mask = rearrange(mask, 'b (h w) -> b 1 h w', h=N_H, w=N_W).float()
        upsampled_mask = F.interpolate(spatial_mask, size=(H, W), mode='nearest')

        x_masked = x * upsampled_mask
        x_in = torch.cat([x_masked, upsampled_mask], dim=1)

        features = self.proj(x_in)
        tokens = rearrange(features, 'b d h w -> b (h w) d')
        return tokens


In [ ]:
# Section 7 & 8: Visual Lp-JEPA & Anti-collapse
class VisualLpJEPA(nn.Module):
    def __init__(self, config: Config):
        super().__init__()
        self.config = config
        self.tokenizer = SCOTTLiteTokenizer(embed_dim=config.visual_embed_dim, patch_size=config.patch_size)

        num_patches = (config.image_size // config.patch_size) ** 2
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches, config.visual_embed_dim) * 0.02)
        self.mask_token = nn.Parameter(torch.randn(1, 1, config.visual_embed_dim) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config.visual_embed_dim, nhead=config.visual_heads,
            dim_feedforward=config.visual_embed_dim * config.visual_mlp_ratio,
            batch_first=True, activation='gelu', norm_first=True
        )
        self.context_encoder = nn.TransformerEncoder(encoder_layer, num_layers=config.visual_depth)
        self.predictor = nn.TransformerEncoder(encoder_layer, num_layers=max(1, config.visual_depth - 1))

        self.target_encoder = nn.TransformerEncoder(encoder_layer, num_layers=config.visual_depth)
        self.target_tokenizer = SCOTTLiteTokenizer(embed_dim=config.visual_embed_dim, patch_size=config.patch_size)
        self._init_target()

    def _init_target(self):
        self.target_encoder.load_state_dict(self.context_encoder.state_dict())
        self.target_tokenizer.load_state_dict(self.tokenizer.state_dict())
        for p in self.target_encoder.parameters(): p.requires_grad = False
        for p in self.target_tokenizer.parameters(): p.requires_grad = False

    def update_target(self):
        decay = self.config.ema_decay
        with torch.no_grad():
            for p_ctx, p_tgt in zip(self.context_encoder.parameters(), self.target_encoder.parameters()):
                p_tgt.data.mul_(decay).add_(p_ctx.data, alpha=1 - decay)
            for p_ctx, p_tgt in zip(self.tokenizer.parameters(), self.target_tokenizer.parameters()):
                p_tgt.data.mul_(decay).add_(p_ctx.data, alpha=1 - decay)

    def forward(self, x: torch.Tensor, mask_ratio: float = 0.5):
        B, C, H, W = x.shape
        N = (H // self.config.patch_size) ** 2

        num_keep = max(1, int(N * (1 - mask_ratio)))
        noise = torch.rand(B, N, device=x.device)
        ids_shuffle = torch.argsort(noise, dim=1)

        mask_binary = torch.zeros(B, N, device=x.device)
        mask_binary.scatter_(1, ids_shuffle[:, :num_keep], 1)

        # 1. Target Forward (Full Image)
        with torch.no_grad():
            tgt_tokens = self.target_tokenizer(x, mask=torch.ones_like(mask_binary))
            tgt_tokens = tgt_tokens + self.pos_embed
            tgt_encoded = self.target_encoder(tgt_tokens)

        # 2. Context Forward (Masked)
        ctx_tokens = self.tokenizer(x, mask=mask_binary)
        ctx_tokens = ctx_tokens + self.pos_embed

        batch_indices = torch.arange(B)[:, None].to(x.device)
        visible_indices = ids_shuffle[:, :num_keep]
        visible_tokens = ctx_tokens[batch_indices, visible_indices]

        encoded_visible = self.context_encoder(visible_tokens)

        # 3. Predictor Forward — pos_embed added to mask tokens only
        # (visible positions already carry positional info via the context encoder)
        pred_tokens = self.mask_token.repeat(B, N, 1) + self.pos_embed
        pred_tokens[batch_indices, visible_indices] = encoded_visible

        predicted_full = self.predictor(pred_tokens)

        masked_indices = ids_shuffle[:, num_keep:]
        pred_masked = predicted_full[batch_indices, masked_indices]
        tgt_masked = tgt_encoded[batch_indices, masked_indices]

        return pred_masked, tgt_masked, encoded_visible, tgt_encoded

def anti_collapse_metrics(features: torch.Tensor):
    """SIGReg-lite anti-collapse metrics. Pass a tensor with grad to use it as a loss term."""
    if features.dim() == 3:
        features = rearrange(features, 'b n d -> (b n) d')

    stds = torch.std(features, dim=0)
    std_mean = stds.mean()
    std_min = stds.min()

    centered = features - features.mean(dim=0, keepdim=True)
    cov = (centered.T @ centered) / (features.shape[0] - 1 + 1e-6)
    cov_offdiag = cov.clone()
    cov_offdiag.fill_diagonal_(0.0)
    cov_offdiag_mean = cov_offdiag.abs().mean()

    # Effective rank — torch.linalg.svd (torch.svd is deprecated)
    _, S, _ = torch.linalg.svd(centered.float()[:1000], full_matrices=False)
    probs = S / (S.sum() + 1e-9)
    entropy = -torch.sum(probs * torch.log(probs + 1e-6))
    eff_rank = torch.exp(entropy)

    return {'std_mean': std_mean, 'std_min': std_min, 'cov_offdiag': cov_offdiag_mean, 'eff_rank': eff_rank}


In [ ]:
# Section 9: Frozen T5 text embedder
class FrozenT5Embedder(nn.Module):
    def __init__(self, model_name="t5-small", max_len=32):
        super().__init__()
        self.tokenizer = T5Tokenizer.from_pretrained(model_name, model_max_length=max_len)
        self.model = T5EncoderModel.from_pretrained(model_name)
        self.max_len = max_len
        self.embed_dim = self.model.config.d_model

        for p in self.model.parameters():
            p.requires_grad = False

    def forward(self, texts: List[str]) -> Tuple[torch.Tensor, torch.Tensor]:
        device = next(self.model.parameters()).device
        encoded = self.tokenizer(texts, padding='max_length', truncation=True,
                                 max_length=self.max_len, return_tensors='pt').to(device)

        with torch.no_grad():
            outputs = self.model(input_ids=encoded.input_ids, attention_mask=encoded.attention_mask)
            token_embs = outputs.last_hidden_state

        mask = encoded.attention_mask.unsqueeze(-1).float()
        pooled_embs = (token_embs * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

        return token_embs, pooled_embs


In [ ]:
# Section 10: Mini-ELF continuous language embedding flow
class MiniELFFlow(nn.Module):
    def __init__(self, config: Config):
        super().__init__()
        self.config = config

        self.time_mlp = nn.Sequential(
            nn.Linear(1, config.mini_elf_width), nn.SiLU(), nn.Linear(config.mini_elf_width, config.mini_elf_width)
        )

        self.cond_proj = nn.Linear(config.visual_embed_dim, config.mini_elf_width)
        self.target_proj = nn.Linear(config.text_embed_dim, config.mini_elf_width)

        layer = nn.TransformerEncoderLayer(
            d_model=config.mini_elf_width, nhead=config.mini_elf_heads,
            dim_feedforward=config.mini_elf_width * 4, batch_first=True, norm_first=True
        )
        self.denoiser = nn.TransformerEncoder(layer, num_layers=config.mini_elf_depth)
        self.out_proj = nn.Linear(config.mini_elf_width, config.text_embed_dim)

    def forward(self, z_t: torch.Tensor, t: torch.Tensor, cond: torch.Tensor):
        t_emb = self.time_mlp(t).unsqueeze(1)
        c_emb = self.cond_proj(cond).unsqueeze(1)
        z_emb = self.target_proj(z_t).unsqueeze(1)

        seq = torch.cat([t_emb, c_emb, z_emb], dim=1)
        out_seq = self.denoiser(seq)

        pred_z = self.out_proj(out_seq[:, 2, :])
        return pred_z

    def sample(self, cond: torch.Tensor, steps=10):
        # Forward Euler in flow-matching time: t=0 (noise) -> t=1 (target).
        # For z_t = t*x1 + (1-t)*x0, the true velocity is (x1 - z) / (1 - t).
        B = cond.shape[0]
        device = cond.device
        z = torch.randn(B, self.config.text_embed_dim, device=device)
        dt = 1.0 / steps

        for i in range(steps):
            t_val = i / steps
            t_tensor = torch.full((B, 1), t_val, device=device)
            with torch.no_grad():
                pred_x = self(z, t_tensor, cond)
            denom = max(1.0 - t_val, dt)
            z = z + dt * (pred_x - z) / denom

        return z


In [ ]:
# Section 11 & 12: VL-JEPA Alignment & Dynamics
class AlignmentModule(nn.Module):
    def __init__(self, config: Config):
        super().__init__()
        self.vis_proj = nn.Linear(config.visual_embed_dim, config.text_bottleneck_dim)
        self.txt_proj = nn.Linear(config.text_embed_dim, config.text_bottleneck_dim)
        self.temperature = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))

    def forward(self, vis_pool: torch.Tensor, txt_pool: torch.Tensor):
        v = F.normalize(self.vis_proj(vis_pool), dim=-1)
        t = F.normalize(self.txt_proj(txt_pool), dim=-1)

        logit_scale = self.temperature.exp().clamp(max=100)
        logits_per_v = logit_scale * v @ t.T
        logits_per_t = logit_scale * t @ v.T
        return logits_per_v, logits_per_t, v, t

class ActionDynamicsModel(nn.Module):
    def __init__(self, config: Config):
        super().__init__()
        self.action_emb = nn.Embedding(config.num_actions, config.visual_embed_dim)
        hidden = config.visual_embed_dim * 2
        depth = max(2, config.dynamics_depth)
        layers = [nn.Linear(config.visual_embed_dim * 2, hidden), nn.SiLU()]
        for _ in range(depth - 2):
            layers += [nn.Linear(hidden, hidden), nn.SiLU()]
        layers += [nn.Linear(hidden, config.visual_embed_dim)]
        self.net = nn.Sequential(*layers)

    def forward(self, z_t: torch.Tensor, action: torch.Tensor):
        a_emb = self.action_emb(action)
        combined = torch.cat([z_t, a_emb], dim=-1)
        return self.net(combined) + z_t


In [ ]:
# Section 13: Full System and Planner
class LpElfJepaSystem(nn.Module):
    def __init__(self, config: Config):
        super().__init__()
        self.config = config
        self.visual = VisualLpJEPA(config)
        self.text = FrozenT5Embedder(config.text_model_name, config.max_text_len)
        self.align = AlignmentModule(config)
        self.mini_elf = MiniELFFlow(config)
        self.dynamics = ActionDynamicsModel(config)

    def encode_vision(self, x: torch.Tensor):
        with torch.no_grad():
            _, _, _, tgt_encoded = self.visual(x, mask_ratio=0.0)
        return tgt_encoded.mean(dim=1)

    def plan(self, current_frame: torch.Tensor, goal_frame: torch.Tensor):
        """Vectorized visual-goal planner. All planner_num_candidates rollouts run as a single
        batched call per dynamics step, so the GPU is actually utilized during planning.
        Energy is cosine distance between rollout-end latent and goal-frame latent."""
        self.eval()
        device = current_frame.device
        with torch.no_grad():
            z_0 = self.encode_vision(current_frame.unsqueeze(0))    # [1, D]
            z_goal = self.encode_vision(goal_frame.unsqueeze(0))    # [1, D]
            z_goal_n = F.normalize(z_goal, dim=-1)                  # [1, D]

            B = self.config.planner_num_candidates
            H = self.config.planner_horizon

            # Sample all action sequences at once
            actions = torch.randint(0, self.config.num_actions, (B, H), device=device)  # [B, H]

            # Broadcast initial state and roll out all candidates in parallel
            z_curr = z_0.expand(B, -1).contiguous()  # [B, D]
            for step in range(H):
                z_curr = self.dynamics(z_curr, actions[:, step])  # [B, D]

            # Cosine energy for every candidate at once
            z_curr_n = F.normalize(z_curr, dim=-1)
            sim = (z_curr_n * z_goal_n).sum(dim=-1)  # [B]
            energies = 1.0 - sim                     # [B]

            # Best candidate
            best_idx = int(energies.argmin().item())
            best_action = int(actions[best_idx, 0].item())
            best_seq = [self.config.action_names[int(a.item())] for a in actions[best_idx]]

            # Per-first-action min energy for the diagnostic bar chart
            candidate_energies: Dict[str, float] = {}
            for a_id in range(self.config.num_actions):
                mask = actions[:, 0] == a_id
                if mask.any():
                    candidate_energies[self.config.action_names[a_id]] = float(energies[mask].min().item())

        return best_action, best_seq, candidate_energies

def select_action_with_diagnostics(frame, goal_frame, system, config):
    action_id, best_seq, energies = system.plan(frame, goal_frame)
    return {
        "action_id": action_id,
        "action_name": config.action_names[action_id],
        "best_sequence": best_seq,
        "candidate_energies": energies,
        "diagnostics": {"best_energy": min(energies.values()) if energies else 0.0}
    }


In [ ]:
# Sections 14-17: Training Stages
def train_stage_1_visual(system: LpElfJepaSystem, dataloader, config: Config):
    optimizer = torch.optim.AdamW(system.visual.parameters(), lr=config.lr, weight_decay=config.weight_decay)
    system.visual.train()

    pbar = tqdm(range(config.stage1_steps), desc="Stage 1: Lp-JEPA")
    iterator = iter(dataloader)

    for step in pbar:
        try: batch = next(iterator)
        except StopIteration: iterator = iter(dataloader); batch = next(iterator)

        frames = batch['frame_t'].to(config.device)
        optimizer.zero_grad()

        with torch.autocast(device_type=config.device, dtype=torch.bfloat16, enabled=config.use_bf16):
            pred_masked, tgt_masked, encoded_vis, tgt_encoded = system.visual(frames, mask_ratio=config.mask_ratio)
            loss_pred = F.smooth_l1_loss(pred_masked, tgt_masked)

            # Anti-collapse regularizer on context encoder output (has gradient)
            metrics_train = anti_collapse_metrics(encoded_vis)
            loss_std = F.relu(1.0 - metrics_train['std_mean'])

            loss = loss_pred * config.lambda_pred + loss_std * config.lambda_std

        loss.backward()
        optimizer.step()
        system.visual.update_target()

        if step % 10 == 0:
            with torch.no_grad():
                metrics_log = anti_collapse_metrics(tgt_encoded)
            pbar.set_postfix(
                l_pred=loss_pred.item(),
                std=metrics_log['std_mean'].item(),
                rnk=metrics_log['eff_rank'].item(),
            )

def train_stage_2_align(system: LpElfJepaSystem, dataloader, config: Config):
    optimizer = torch.optim.AdamW(system.align.parameters(), lr=config.lr)
    system.align.train()

    pbar = tqdm(range(config.stage2_steps), desc="Stage 2: Alignment")
    iterator = iter(dataloader)

    for step in pbar:
        try: batch = next(iterator)
        except StopIteration: iterator = iter(dataloader); batch = next(iterator)

        frames = batch['frame_t'].to(config.device)
        texts = batch['state_text']

        optimizer.zero_grad()
        with torch.autocast(device_type=config.device, dtype=torch.bfloat16, enabled=config.use_bf16):
            vis_pool = system.encode_vision(frames)
            _, txt_pool = system.text(texts)

            logits_v, logits_t, _, _ = system.align(vis_pool, txt_pool)
            labels = torch.arange(frames.shape[0], device=config.device)
            loss_nce = (F.cross_entropy(logits_v, labels) + F.cross_entropy(logits_t, labels)) / 2

        loss_nce.backward()
        optimizer.step()

        if step % 10 == 0:
            pbar.set_postfix(nce=loss_nce.item())

def train_stage_3_minielf(system: LpElfJepaSystem, dataloader, config: Config):
    optimizer = torch.optim.AdamW(system.mini_elf.parameters(), lr=config.lr)
    system.mini_elf.train()

    pbar = tqdm(range(config.stage3_steps), desc="Stage 3: Mini-ELF")
    iterator = iter(dataloader)

    for step in pbar:
        try: batch = next(iterator)
        except StopIteration: iterator = iter(dataloader); batch = next(iterator)

        frames = batch['frame_t'].to(config.device)
        texts = batch['rationale_text']

        optimizer.zero_grad()
        with torch.autocast(device_type=config.device, dtype=torch.bfloat16, enabled=config.use_bf16):
            vis_pool = system.encode_vision(frames)
            _, target_txt = system.text(texts)

            noise = torch.randn_like(target_txt)
            t = torch.rand((target_txt.shape[0], 1), device=config.device)
            z_t = t * target_txt + (1 - t) * noise

            pred_txt = system.mini_elf(z_t, t, cond=vis_pool)
            loss_flow = F.mse_loss(pred_txt, target_txt)

        loss_flow.backward()
        optimizer.step()
        if step % 10 == 0: pbar.set_postfix(flow=loss_flow.item())

def train_stage_4_dynamics(system: LpElfJepaSystem, dataloader, config: Config):
    optimizer = torch.optim.AdamW(system.dynamics.parameters(), lr=config.lr)
    system.dynamics.train()

    pbar = tqdm(range(config.stage4_steps), desc="Stage 4: Dynamics")
    iterator = iter(dataloader)

    for step in pbar:
        try: batch = next(iterator)
        except StopIteration: iterator = iter(dataloader); batch = next(iterator)

        f_t = batch['frame_t'].to(config.device)
        f_tp1 = batch['frame_tp1'].to(config.device)
        actions = batch['action_id'].to(config.device)

        optimizer.zero_grad()
        with torch.autocast(device_type=config.device, dtype=torch.bfloat16, enabled=config.use_bf16):
            z_t = system.encode_vision(f_t)
            z_tp1 = system.encode_vision(f_tp1).detach()

            z_tp1_hat = system.dynamics(z_t, actions)
            loss_dyn = F.smooth_l1_loss(z_tp1_hat, z_tp1)

            copy_loss = F.smooth_l1_loss(z_t, z_tp1).item()

        loss_dyn.backward()
        optimizer.step()
        if step % 10 == 0: pbar.set_postfix(dyn=loss_dyn.item(), cp=copy_loss)


In [ ]:
# Section 18: Smoke Tests + Training Pipeline
print("Running Smoke Tests & Initializing System...")
system = LpElfJepaSystem(config).to(config.device)

# Report params + initial VRAM
_n_train = sum(p.numel() for p in system.parameters() if p.requires_grad)
_n_total = sum(p.numel() for p in system.parameters())
print(f"Trainable params: {_n_train/1e6:.1f}M  |  Total params: {_n_total/1e6:.1f}M")
if config.device == 'cuda':
    torch.cuda.reset_peak_memory_stats()

sample = next(iter(train_loader))
f_test = sample['frame_t'].to(config.device)
assert system.encode_vision(f_test).shape == (config.batch_size, config.visual_embed_dim)
print("Forward passes passed.")

if config.device == 'cuda':
    torch.cuda.synchronize()
    print(f"VRAM after one forward: allocated={torch.cuda.memory_allocated()/1e9:.2f} GB, peak={torch.cuda.max_memory_allocated()/1e9:.2f} GB")

train_stage_1_visual(system, train_loader, config)
if config.device == 'cuda':
    print(f"VRAM after Stage 1: peak={torch.cuda.max_memory_allocated()/1e9:.2f} GB")
train_stage_2_align(system, train_loader, config)
train_stage_3_minielf(system, train_loader, config)
train_stage_4_dynamics(system, train_loader, config)
if config.device == 'cuda':
    print(f"VRAM final peak: {torch.cuda.max_memory_allocated()/1e9:.2f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("Training Complete!")


Running Smoke Tests & Initializing System...


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/51 [00:00<?, ?it/s]

Trainable params: 59.6M  |  Total params: 120.7M
Forward passes passed.
VRAM after one forward: allocated=0.57 GB, peak=1.48 GB


Stage 1: Lp-JEPA:   0%|          | 0/6000 [00:00<?, ?it/s]

In [ ]:
# Section 19: 2.5D / TopDown Trial Ground (matplotlib renderer)
class VoxelTrialGroundEnv:
    """Env returns tensors at model input resolution (config.image_size); RGB output
    is upscaled to config.render_size only for video / display purposes."""
    def __init__(self, config: Config):
        self.config = config
        self.base_env = BlockWorldEnv(size=config.grid_size, image_size=config.image_size)
        self.render_size = config.render_size

    def reset(self, seed: Optional[int]=None, goal_text: Optional[str]=None):
        frame = self.base_env.reset(seed)
        self.current_goal = goal_text if goal_text else "reach the green goal"
        self.step_count = 0
        return self._build_obs(frame)

    def step(self, action_id: int):
        frame, success, gt, st, rt = self.base_env.step(action_id)
        self.step_count += 1
        done = success or self.step_count >= self.config.trial_max_steps
        obs = self._build_obs(frame)
        return obs, 1.0 if success else 0.0, done, False, {'rationale': rt, 'success': success}

    def _build_obs(self, frame_tensor):
        rgb_array = (rearrange(frame_tensor, 'c h w -> h w c').numpy() * 255).astype(np.uint8)
        # Upscale ONLY for visualization; keep the tensor at model resolution.
        scale = max(1, self.render_size // self.config.image_size)
        if scale > 1:
            rgb_render = rgb_array.repeat(scale, axis=0).repeat(scale, axis=1)
        else:
            rgb_render = rgb_array
        # Goal frame (success state) — what success would look like.
        goal_tensor = self.base_env.get_goal_frame()
        goal_rgb = (rearrange(goal_tensor, 'c h w -> h w c').numpy() * 255).astype(np.uint8)
        if scale > 1:
            goal_rgb = goal_rgb.repeat(scale, axis=0).repeat(scale, axis=1)
        return {
            'agent_view_rgb': rgb_render,
            'topdown_rgb': rgb_render,
            'tensor': frame_tensor,           # model-resolution, for encoding
            'goal_tensor': goal_tensor,       # model-resolution goal frame, for planner
            'goal_rgb': goal_rgb,             # rendered goal frame, for display
            'goal_text': getattr(self, 'current_goal', 'reach the green goal'),
        }

    def render(self, mode="rgb_array"):
        return self._build_obs(self.base_env._get_obs())['topdown_rgb']

# Construct latent bank
latent_bank = []
system.eval()
with torch.no_grad():
    for batch in val_loader:
        f = batch['frame_t'].to(config.device)
        z = system.encode_vision(f).cpu()
        for i in range(len(f)):
            latent_bank.append({'frame': f[i].cpu(), 'latent': z[i]})
        if len(latent_bank) > 200: break

def get_nearest_future(z_pred):
    dists = [F.mse_loss(z_pred.cpu(), lb['latent'].unsqueeze(0)).item() for lb in latent_bank]
    best_idx = np.argmin(dists)
    return rearrange(latent_bank[best_idx]['frame'], 'c h w -> h w c').numpy()


In [ ]:
# Section 20 & 21: Trial Modes and A/B Comparison
def run_autonomous_trial(env, system, config, seed=101):
    obs = env.reset(seed=seed)
    frames = [obs['topdown_rgb']]
    done = False
    info = {'success': False}

    while not done:
        frame_t = obs['tensor'].to(config.device)
        goal_frame = obs['goal_tensor'].to(config.device)
        result = select_action_with_diagnostics(frame_t, goal_frame, system, config)
        obs, rew, done, _, info = env.step(result['action_id'])
        frames.append(obs['topdown_rgb'])

    return frames, info['success']

print("Running A/B Comparison Arena: Latent Planner vs Random Policy")
planner_successes = 0
random_successes = 0
num_trials = 20
env = VoxelTrialGroundEnv(config)

for i in range(num_trials):
    # Random
    env.reset(seed=1000+i)
    done = False
    info = {'success': False}
    while not done:
        _, rew, done, _, info = env.step(np.random.randint(0, config.num_actions))
    random_successes += int(info['success'])

    # Planner
    _, success = run_autonomous_trial(env, system, config, seed=1000+i)
    planner_successes += int(success)

print(f"Random Policy Success Rate:  {random_successes / num_trials:.0%}")
print(f"Latent Planner Success Rate: {planner_successes / num_trials:.0%}")

# Record and show an MP4 of the planner
frames, _ = run_autonomous_trial(env, system, config, seed=42)
video_path = os.path.join(config.output_dir, "planner_trial.mp4")
imageio.mimsave(video_path, frames, fps=2)

print("Displaying Planner Trajectory Video:")
display(Video(video_path, embed=True, width=256))


In [ ]:
# Sections 22, 23, 24: Visualizations, Diagnostics and Next Steps
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
obs = env.reset(seed=99)

frame_t = obs['tensor'].to(config.device)
goal_frame = obs['goal_tensor'].to(config.device)
result = select_action_with_diagnostics(frame_t, goal_frame, system, config)

z_0 = system.encode_vision(frame_t.unsqueeze(0))
z_1 = system.dynamics(z_0, torch.tensor([result['action_id']], device=config.device))
nearest_frame = get_nearest_future(z_1)

axes[0].imshow(obs['topdown_rgb'])
axes[0].set_title("Current Frame")
axes[0].axis('off')

axes[1].imshow(obs['goal_rgb'])
axes[1].set_title("Goal Frame\n(success state)")
axes[1].axis('off')

acts, energies = list(result['candidate_energies'].keys()), list(result['candidate_energies'].values())
axes[2].bar(acts, energies, color='skyblue')
axes[2].set_title("Action Energies\n(Lower = Closer to Goal)")
axes[2].tick_params(axis='x', rotation=45)

axes[3].imshow(nearest_frame)
axes[3].set_title(f"Nearest Predicted Future\nAction: {result['action_name']}")
axes[3].axis('off')

plt.tight_layout()
plt.show()

print("\n--- Final Diagnosis and Limitations ---")
print(f"Active config: {config.config_name}")
print("1. This is a toy narrow-domain prototype. It does not train a general LLM.")
print("2. The Mini-ELF flow demonstrates condition-based language semantic shifting, avoiding slow autoregressive decoding.")
print("3. Planner is now visual-goal-conditioned: it compares rollout-end latents to the latent of a goal frame.")
print("4. Visual-language alignment (Stage 2) is still trained as a separate research demonstration but no longer drives planning.")
print("5. Limitations: BlockWorld physics are trivial. The next logical step is moving this architecture onto embodied datasets like MineDojo or real robotic trajectories.")
